**Self-Intro**

In [ ]:
Hello, I am Rakesh.

I have 7+ years of experience in IT, out of which 4+ years are focused specifically on Azure and 1 year on Microsoft Fabric,
working across diverse domains like Banking, Retail, Finance, and Automotive.

In my most recent project with ALD Automotive two business line TCO and B3M, my responsibility is to design, develop,
and optimize data pipelines using the Medallion Architecture on both Azure and Fabric platforms.
I work on ingesting data from various upstream source systems like SQL Server, SFTP, and ADLS Gen2 into the Bronze Layer will keep RAW data.
In the Silver Layer, I handle data preprocessing, quality checks, standardization, and maintain historical data using SCD Type 2 patterns.

Once data is cleansed and validated, I move to the Gold Layer, where I aggregate and apply business logic —
such as Cost Per Kilometer calculations, contract profitability scoring,
and excess mileage projections — storing curated data for downstream consumption.
I build batch pipelines using Azure Data Factory, perform transformations using PySpark and Spark SQL,
and have implemented a metadata-driven validation framework for data quality and governance.

One of my key achievements was being part of the migration of the TCO business line
from Azure to Microsoft Fabric — — this involved re-architecting pipelines to leverage OneLake for unified storage,
Data Pipelines for orchestration, and Fabric Notebooks for PySpark transformations,
:resulting in improved performance and simplified governance.

My core tech stack includes: Azure Data Factory, Databricks, Synapse Analytics, ADLS Gen2, Delta Lake, PySpark, Spark SQL,
 and Microsoft Fabric (Lakehouse, Notebooks, Data Pipelines, and Power BI with Direct Lake)

I have strong experience in building scalable, production-grade data solutions in both Azure and Fabric.

**SDC Type-2 Implementataion**

In [ ]:
-- STEP 1: Expire changed records
MERGE INTO dim_vehicle t
USING source_vehicle s
ON t.vehicle_id = s.vehicle_id AND t.is_current = true  #business key

WHEN MATCHED AND (t.make <> s.make OR t.model <> s.model OR t.fuel_type <> s.fuel_type)
THEN UPDATE SET
    t.end_date = current_date(),
    t.is_current = false;

-- STEP 2: Insert new versions + new records
MERGE INTO dim_vehicle t
USING source_vehicle s
ON t.vehicle_id = s.vehicle_id AND t.is_current = true

WHEN NOT MATCHED THEN INSERT (
    vehicle_id, make, model, fuel_type, start_date, end_date, is_current
)
VALUES (
    s.vehicle_id, s.make, s.model, s.fuel_type, current_date(), NULL, true
);


#============================================
# SCD TYPE-2: dim_vehicle
#============================================
# BUSINESS KEY:     vehicle_id (unique identifier)
# TRACKING COLS:    make, model, fuel_type (changes trigger new version)
# SCD COLS:         start_date, end_date, is_current (history tracking)
# ============================================

#STEP 1: Expire changed records (close old version)
MERGE INTO dim_vehicle t
USING source_vehicle s
ON t.vehicle_id = s.vehicle_id       # 🔑 BUSINESS KEY
   AND t.is_current = true           # 📌 Only match current records

WHEN MATCHED AND (
    t.make <> s.make OR              # 🔄 TRACKING COLUMN
    t.model <> s.model OR            # 🔄 TRACKING COLUMN
    t.fuel_type <> s.fuel_type       # 🔄 TRACKING COLUMN
)
THEN UPDATE SET
    t.end_date = current_date(),     # 📅 SCD COLUMN (close record)
    t.is_current = false;            # 📅 SCD COLUMN (mark as history)

#STEP 2: Insert new versions + completely new records
MERGE INTO dim_vehicle t
USING source_vehicle s
ON t.vehicle_id = s.vehicle_id       # 🔑 BUSINESS KEY
   AND t.is_current = true

WHEN NOT MATCHED THEN INSERT (
    vehicle_id,                      # 🔑 BUSINESS KEY
    make, model, fuel_type,          # 🔄 TRACKING COLUMNS
    start_date, end_date, is_current # 📅 SCD COLUMNS
)
VALUES (
    s.vehicle_id,                    # 🔑 BUSINESS KEY
    s.make, s.model, s.fuel_type,    # 🔄 TRACKING COLUMNS
    current_date(), NULL, true       # 📅 SCD COLUMNS (new active record)
);

**End To End Pipeline**

In [ ]:
============================================================
ADF SQL Framework Flow (Full + Incremental + Backfill + History)
============================================================
""" Pre-requist -
01.ON-prim connection onprim-gate way/SHIR
02.Metadata tbl -> to define what to load and How to load
03.Control tbl -> for tracking when and status of ingestion
"""

1) Lookup Activity (Config Table)
   Use: Get list of tables + load type (FULL/INCR/BACKFILL)

2) ForEach Activity (Iterate Tables)
   Use: Loop through each table one by one

3) If Condition Activity (Full Load Check)
   Use: Decide whether table needs initial history load
   -
   L → Run full extract
   - Else → Go incremental path

4) Lookup Activity (Watermark Table)
   Use: Fetch last processed value for current table

5) If Condition Activity (Backfill Check)
   Use: Decide whether to reprocess last N days or only new delta

6) Copy Activity (SQL → ADLS Bronze)
   Use: Load data from SQL source into raw zone
   - FULL: All rows
   - INCR: Only rows > watermark
   - BACKFILL: Rows from last N days

7) Notebook / Stored Procedure Activity (Merge History)
   - to update the water mark column and full the audit table

8) Web activity
-for send mail allert using logic app






In [ ]:
SFTP  (100 small product files)
   │
   ▼
[ Get Metadata Activity ]
Purpose:
Fetch list of all files dynamically from SFTP folder
   │
   ▼
[ Filter Activity ]
Purpose:
Keep only required files (ex: .csv files)
   │
   ▼
[ ForEach Activity  (Parallel = 20) ]
Purpose:
Process multiple files in parallel for faster ingestion
   │
   ▼
[ Copy Activity  (SFTP → ADLS Landing) ]
Purpose:
Move files as-is into Landing zone (no transformation)
   │
   ▼
[ Databricks Notebook Activity ]
Purpose:
Trigger Spark job to solve small file problem
   │
   ▼
========== DATABRICKS ==========
Read landing folder in parallel
Extract product from file path
Repartition by product (parallel processing)
Write Partitioned Delta files
Run OPTIMIZE (file compaction)
================================
   │
   ▼
BRONZE LAYER (Optimized Delta Files)


**100 small product files in ADLS Gen2**

In [ ]:
# -------------------------------
# Small Files Batch Processing Demo
# Scenario: 100 small product files in ADLS Gen2
# Goal: Read -> Enrich -> Repartition -> Write Delta -> Optimize
# -------------------------------

# 1) Enable automatic file optimization (production best practice)
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled","true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled","true")

# 2) Define paths
landing_path = "abfss://datalake@storage.dfs.core.windows.net/landing/products/"
bronze_path  = "abfss://datalake@storage.dfs.core.windows.net/bronze/products"

# 3) Read ALL product files in parallel (No looping)
df = spark.read.format("csv") \
    .option("header","true") \
    .load(landing_path)

# 4) Extract product name from folder path
from pyspark.sql.functions import input_file_name, regexp_extract

df = df.withColumn("file_path", input_file_name())

df = df.withColumn(
    "product",
    regexp_extract("file_path", "product=([^/]+)", 1)
)

# 5) Repartition by product to fully utilize cluster
df = df.repartition(100, "product")

# 6) Write as Partitioned Delta table (Compaction step)
df.write.format("delta") \
    .mode("append") \
    .partitionBy("product") \
    .save(bronze_path)

# 7) Run OPTIMIZE to merge remaining small files (maintenance job)
spark.sql(f"""
OPTIMIZE delta.`{bronze_path}`
ZORDER BY (product)
""")

print("Batch processing completed successfully!")


**100+ small product files (KB size) daily in SFTP**

In [ ]:
SCENARIO:
We receive 100+ small product files (KB size) daily in SFTP.
Goal is to build an optimized ADF pipeline to ingest data into Bronze layer in ADLS and remove small file problem.

----------------------------------------------------
END-TO-END PIPELINE EXPLANATION
----------------------------------------------------

1) First, ADF connects to SFTP, ADLS Gen2 and Azure Databricks using Linked Services.

2) Pipeline starts with Get Metadata activity to list all files from the SFTP folder dynamically.
   This makes the pipeline metadata-driven and avoids hardcoding file names.

3) Next, we use Filter activity to select only required files such as CSV.
   This ensures unwanted files are not processed.

4) Then we use ForEach activity to copy files in parallel from SFTP to ADLS Landing zone.
   We configure batch count around 20 to enable parallel ingestion and speed up SFTP reads.

5) Copy activity moves files exactly as-is into Landing zone using dynamic folder path based on date.
   ADF is used only for ingestion and not for heavy transformation.

6) After ingestion completes, ADF triggers a Databricks notebook for small file optimization.
   This is the key step to solve the small file problem.

7) Databricks reads the entire landing folder in one Spark read instead of processing file-by-file.
   Spark automatically distributes files across executors for parallel processing.

8) We extract product name from file path and repartition the dataframe by product.
   This ensures full cluster utilization and balanced processing.

9) Data is written into Bronze layer as a partitioned Delta table.
   This compacts hundreds of small files into few large optimized files (100–500MB).

10) Auto Optimize and Auto Compact are enabled and an OPTIMIZE command runs to merge remaining small files.
    This ensures fast downstream queries and long-term performance.

----------------------------------------------------
FINAL FLOW:
SFTP → ADF Parallel Copy → ADLS Landing → Databricks Compaction → Delta Bronze

This approach solves the small file problem, improves cluster utilization, and builds a scalable batch ingestion pipeline.


**Row Level Security (RLS)**

In [ ]:
#Row Filter Function Using Group Membership

#step :1
CREATE FUNCTION main_catalog.company.group_rls(dept STRING)
RETURN
  CASE
    WHEN is_member('hr_team') AND dept = 'HR' THEN TRUE
    WHEN is_member('finance_team') AND dept = 'Finance' THEN TRUE
    ELSE FALSE
  END;
#step :2
ALTER TABLE main_catalog.company.employee
SET ROW FILTER main_catalog.company.group_rls ON (dept);


#Row Filter Function Using Current User
#step :1
CREATE FUNCTION main_catalog.company.user_rls(dept STRING)
RETURN
  CASE
    WHEN current_user() = 'ravi@datacorp.com' AND dept = 'HR'
    THEN TRUE
    ELSE FALSE
  END;

#Step :2
ALTER TABLE main_catalog.company.employee
SET ROW FILTER main_catalog.company.user_rls ON (dept);



**Column Level Security (CLS)**

In [ ]:
#GROUP LEVEL
#Step :1
CREATE FUNCTION main_catalog.company.group_mask_salary(salary INT)
RETURN
  CASE
    WHEN is_member('finance_team')
    THEN salary
    ELSE NULL
  END;

#Step :2
ALTER TABLE employee
ALTER COLUMN salary
SET MASK main_catalog.company.group_mask_salary;

#User Level
#Step :1
CREATE FUNCTION main_catalog.company.user_mask_salary(salary INT)
RETURN
  CASE
    WHEN current_user() = 'john@datacorp.com'
    THEN salary
    ELSE NULL
  END;

#Step :2
ALTER TABLE employee
ALTER COLUMN salary
SET MASK main_catalog.company.user_mask_salary;


**Schema Drift**

In [ ]:

Task:
• Handle schema drift without breaking ingestion or impacting reporting.

Action:
• Allowed flexible raw ingestion in the Bronze layer to land data successfully.
• Enforced strict schema validation before loading into Silver and Gold layers.
• Used a schema registry + metadata checks to compare expected vs actual schema.
• Flagged unexpected changes, logged them, and stopped corrupted data from moving forward.

Result:
• Pipeline failures reduced, data quality improved, and TCO reporting stayed consistent despite evolving source schemas.


**Late Arrival Files**

In [ ]:
Task:
• Ensure late files are ingested automatically without manual reruns.

Action:
• Built a watermark-based ingestion framework in Azure Data Factory.
• Maintained a control table in Azure SQL to store last processed business date.
• Used Lookup activity to fetch the watermark and process pending dates dynamically.
• Added Get Metadata + If Condition to validate file availability before processing.
• Logged missing files and triggered alerts instead of failing pipelines.

Result:
• Late files were picked up automatically, data completeness improved, and TCO reports stayed accurate with less operational effort.

# ADF- Azure datafactory

**Tracking Failures and re-try**

In [ ]:
"In our Azure Data Engineering platform, I tracked ingestion failures using
• Built a centralized audit logging framework for all ADF pipelines.
• Logged each run into an audit table (source, file/table, status, rows loaded, error details).
• Used On Failure activity paths to capture and store errors immediately.
• Configured retry policies in Copy activities (2–3 retries with intervals) for temporary issues.
• Triggered email/Teams alerts when failures exceeded retry limits.
• Provided clear visibility, faster troubleshooting, and reduced pipeline downtime.

**how you will pass parameter in runtime**

In [ ]:
"In ADF, I pass runtime parameters by defining pipeline parameters like
table name or file name. The master pipeline reads these values from a
config table using a Lookup activity and loops through them with ForEach.
Then it passes the values to Copy or child pipelines, so the same pipeline
can load different tables or files without creating separate pipelines."

**End to end pipeline with files**

In [ ]:
Sure. Below is a real-world enterprise design for SFTP/ADLS file ingestion
with multiple file formats, date-wise folders, and a config-driven framework
(ALD Automotive TCO style).

============================================================
1. Real-World SFTP Folder Structure (Date-wise Delivery)
============================================================

Upstream systems drop daily files like this:

SFTP Root: /ald_tco/

------------------------------------------------------------
/ald_tco/vehicle_cost/2026/01/29/
    vehicle_cost_20260129.csv

/ald_tco/fuel_transactions/2026/01/29/
    fuel_txn_20260129.json

/ald_tco/maintenance_invoice/2026/01/29/
    maint_inv_20260129.parquet
------------------------------------------------------------

3 Different File Formats:
- CSV  (vehicle cost)
- JSON (fuel transactions)
- Parquet (maintenance invoices)

============================================================
2. Config Table (Drives All File Pipelines)
============================================================

Table Name: file_ingestion_config

---------------------------------------------------------------------------------------------------
source_name     source_folder              file_pattern              file_type   target_table
---------------------------------------------------------------------------------------------------
vehicle_cost    /ald_tco/vehicle_cost/     vehicle_cost_*.csv        CSV         silver_vehicle_cost
fuel_txn        /ald_tco/fuel_transactions/ fuel_txn_*.json          JSON        silver_fuel_txn
maint_invoice   /ald_tco/maintenance_invoice/ maint_inv_*.parquet   PARQUET     silver_maintenance
---------------------------------------------------------------------------------------------------

Additional Columns (Enterprise):

---------------------------------------------------------------------------------------------------
date_partitioned   expected_daily   validation_required
---------------------------------------------------------------------------------------------------
Y                  Y                Y
Y                  Y                Y
Y                  Y                N
---------------------------------------------------------------------------------------------------

============================================================
3. Processed File Log Table (Avoid Duplicates + Late Files)
============================================================

Table: processed_file_log

---------------------------------------------------------------------------------------------------
source_name      file_name                    processed_date     status
---------------------------------------------------------------------------------------------------
vehicle_cost     vehicle_cost_20260128.csv    2026-01-28         SUCCESS
fuel_txn         fuel_txn_20260128.json       2026-01-28         SUCCESS
---------------------------------------------------------------------------------------------------

============================================================
4. Parameterized ADF Pipeline Flow (Real World)
============================================================

Pipeline Name: PL_File_Master_Ingestion

Pipeline Parameter:
- p_source_name (optional, else process all)

------------------------------------------------------------
Activity Flow
------------------------------------------------------------

1) Lookup Activity (Read Config)
   Use: Get all file sources to ingest today

   Query:
   SELECT * FROM file_ingestion_config

        |
        v

2) ForEach Activity (Loop Each Source)
   Use: Iterate vehicle_cost, fuel_txn, maint_invoice

        |
        v

3) Set Variable (Build Today Folder Path)
   Use: Construct dynamic date-wise path

   Example Expression:
   @concat(
     item().source_folder,
     formatDateTime(utcNow(),'yyyy/MM/dd'),
     '/'
   )

   Output:
   /ald_tco/vehicle_cost/2026/01/29/

        |
        v

4) Get Metadata Activity (List Files)
   Use: List files in today’s folder

        |
        v

5) Lookup Activity (Processed File Log)
   Use: Fetch already processed files for this source

        |
        v

6) Filter Activity (New/Late Files Only)
   Use: Remove files already processed
        Keep only unprocessed arrivals

        |
        v

7) ForEach Activity (Loop Each File)

   Inside:

   7a) Copy Activity (SFTP → Bronze ADLS)
       Use: Land raw file into Bronze zone

       Target Path:
       /bronze/@{item().source_name}/

   7b) Notebook Activity (Format-Specific Load)
       Use: Read based on file_type

       If CSV → spark.read.csv()
       If JSON → spark.read.json()
       If PARQUET → spark.read.parquet()

       Write into Silver table:
       silver_vehicle_cost, silver_fuel_txn, etc.

   7c) Log File Processed
       Use: Insert into processed_file_log

        |
        v

8) If Condition (Missing File Alert)
   Use: If expected_daily = Y and no file found → send alert

        |
        v

[End]

============================================================
5. How Late Arriving Files Are Handled
============================================================

Example:
vehicle_cost_20260129.csv arrives late on 30th Jan.

Next pipeline run:
- Get Metadata lists file
- Filter checks processed_file_log
- File not processed → picked automatically

No manual rerun needed.

============================================================
One Line Summary
============================================================

Config-driven source list → Build date folder → List files →
Filter unprocessed → Copy to Bronze → Load to Silver → Log processed →
Alert if missing

============================================================

If you want, I can give the exact Filter expression logic used in ADF
to compare Get Metadata output with processed_file_log.


**END to END automation**

In [ ]:
Interview answer — "What automation have you built in this project?"
1. Fully parameterized pipeline — zero hardcoding:
Every single value — connection strings, ADLS paths, table names, DQ thresholds, business rates — lives in one config.json file stored in ADLS.
 All Databricks notebooks accept parameters via widgets. ADF injects pipeline_run_id, run_date, load_mode, and config_path into every notebook automatically.
 To onboard a new environment or change a threshold, you update one file. No notebook, no ADF pipeline, no code file contains a hardcoded value.

2. Scheduled daily trigger
ADF fires the master orchestrator at 01:00 AM W. Europe Standard Time via a cron schedule. The run date is auto-computed inside ADF as formatDateTime(utcNow(), 'yyyy-MM-dd')
 and passed as a parameter, so historical re-runs for any past date are trivially done by manually triggering with a different date.

3. Parallel Bronze ingestion
The three Bronze notebooks — SQL, SFTC, Gen2 — run simultaneously in ADF using parallel activity wiring. Silver only starts after all three succeed.
This is a key performance automation — without it you'd run them one after another and waste time. Failure in any one source gets caught and notified while the others still complete.

4. Retry and timeout policy
Bronze activities are configured for 2 retries with a 60-second wait. Silver and Gold get 1 retry with 120-second wait. This handles transient failures — cluster startup delays, network blips on SFTP, SAS token throttling — without any manual intervention.
5. Watermark-based incremental load
The WatermarkManager class maintains a Delta control table tracking the last-loaded high-watermark value per source and table. Every SQL run reads only rows where last_updated_dt > last_watermark. On a successful run, the watermark advances to the max value seen. A full reload is triggered by passing load_mode=full as a parameter — no code change needed.
6. Schema drift detection and quarantine
On every SQL load, the incoming column set is compared to a stored reference schema JSON file in ADLS. If columns are added or removed, a drift event is logged. The default action is quarantine — the drifted batch is written to a separate rejected Delta path so the main Bronze table stays clean. The team gets alerted and can decide whether to update the reference schema and replay.
7. Corrupt record quarantine for SFTC and Gen2
CSV and JSON file reads use Spark's PERMISSIVE mode. Bad rows land in a _corrupt_record column. After the read, good and bad rows are split: clean rows go to Bronze Delta, corrupt rows go to rejected/corrupt_records/{source}/{run_date}/ as a separate Delta table. Available for reprocessing without a full reload. The rejection count is captured in the audit log.
8. SFTP archive after load
After each SFTP file is staged to ADLS, it is automatically moved to the archive/ folder on the SFTP server. This is controlled by the archive_after_load flag in config. Prevents double-processing on re-runs without needing a separate file tracking registry.
9. Automated DQ profiling on every Silver run
Every Silver table is profiled automatically on each run: null percentage per column, duplicate row count on primary key columns, and min/max/mean on numeric fields. Results are written to silver/_profiling/{table}/{date}/profile.json for a permanent audit trail. Any breach of configured thresholds — say, null% exceeding 5% — is written to a _dq_results Delta table and immediately triggers a DQ alert email via Logic App.
10. SCD Type-2 with hash-based change detection
Dimension tables — dim_vehicle, dim_customer, dim_contract — are managed using SCD Type-2. Rather than comparing every column individually, an MD5 hash is computed across all tracked columns. On a hash mismatch, the old row is expired (scd_end_date = today, is_current = false) and a new row is inserted (scd_start_date = today, is_current = true). New rows with no match in the target are simply inserted as new active records. Delta VACUUM runs automatically post-cycle keeping 7 days of version history.
11. Delta OPTIMIZE and ZORDER post-write
After every Silver table write, OPTIMIZE ZORDER BY (primary_key) runs automatically to co-locate data for fast downstream queries. After every Gold table write, OPTIMIZE runs to compact small files. At the cluster level, optimizeWrite, autoCompact, and adaptive query execution are all enabled. This means no manual compaction jobs are needed — the storage stays query-efficient automatically.
12. Full audit logging in Delta
AuditLogger writes a RUNNING record at the start of every notebook-table combination. On success it updates with records_read, records_written, records_rejected, and the end timestamp. On failure it stamps FAILED with the full error message. The table bronze_tco.pipeline_audit_log is partitioned by run_date so SLA queries across any date range run efficiently. This feeds directly into an operational monitoring dashboard.
Email notifications via Logic App (3 types)
EmailNotifier fires an HTTP POST to a Logic App webhook URL fetched from Key Vault. Three distinct alert types: (1) Success — fired after Gold completes, includes records read/written and total duration. (2) Failure — fired immediately on any layer failure, includes the failed layer, table name, error message, and a direct reference to the audit log table. (3) DQ alert — fired per-table when any profiling rule is breached, with the rule name, actual value, and threshold in an HTML table. Logic App sends via Office 365 with high-priority flag set on failure emails.
Key Vault — zero secrets in code
All 14 secrets — SQL password, SFTP private key, Gen2 SAS token, Logic App webhook URL, Databricks personal access token — are stored exclusively in Azure Key Vault. Databricks accesses them via an AKV-backed secret scope using dbutils.secrets.get(scope, key). The KeyVaultClient class caches secrets in-session to avoid repeated API calls. Secrets never appear in notebook code, ADF pipelines, config files, or logs — only their reference names are stored.